In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Read Bronze JSON files

In [0]:
BRONZE = "abfss://bronze@logisticdatalakestorage.dfs.core.windows.net/"
SILVER = "abfss://silver@logisticdatalakestorage.dfs.core.windows.net/openweathermap/"

### Build City Reference from config.json

In [0]:
df_cities_raw = spark.read.option('multiline', 'true').json(BRONZE + 'config/config.json')

df_city_ref = df_cities_raw \
    .select(explode(col('cities')).alias('c')) \
    .select(
        initcap(col('c.name')).alias('mapped_city'),
        upper(col('c.country')).alias('mapped_country'),
        col('c.lat').alias('c_lat'),
        col('c.lon').alias('c_lon')
    )

print(f'City reference rows: {df_city_ref.count()}')
df_city_ref.display()

City reference rows: 10


mapped_city,mapped_country,c_lat,c_lon
Mumbai,IN,19.076,72.8777
Berlin,DE,52.52,13.405
London,GB,51.5074,-0.1278
New York,US,40.7128,-74.006
Bangkok,TH,13.7563,100.5018
Dubai,AE,25.2048,55.2708
Singapore,SG,1.3521,103.8198
Sydney,AU,-33.8688,151.2093
Paris,FR,48.8566,2.3522
Sao Paulo,BR,-23.5505,-46.6333


In [0]:
df_raw = spark.read \
    .option('multiline', 'true') \
    .option('mode', 'PERMISSIVE') \
    .json(BRONZE + 'openweathermap/')

# _corrupt_record column only exists when Spark finds bad rows
# Check first before trying to filter — otherwise UNRESOLVED_COLUMN error
if '_corrupt_record' in df_raw.columns:
    df_clean = df_raw.filter(col('_corrupt_record').isNull()).drop('_corrupt_record')
    print('Corrupt records found and filtered.')
else:
    df_clean = df_raw
    print('No corrupt records — all rows clean.')

print(f'Bronze rows: {df_clean.count()}')
df_clean.limit(2).display()

No corrupt records — all rows clean.
Bronze rows: 10


base,clouds,cod,coord,dt,id,main,name,rain,sys,timezone,visibility,weather,wind,year,month,day,hour
stations,List(68),200,"List(13.7563, 100.5018)",1779884303,1608132,"List(38.76, 1006, 73, 1006, 1006, 31.76, 32.95, 31.07)",Nonthaburi,null,"List(TH, 2105433, 1779835785, 1779882048, 2)",25200,10000,"List(List(broken clouds, 04n, 803, Clouds))","List(192, 8.09, 5.46)",2026,5,27,12
stations,List(40),200,"List(-23.5505, -46.6333)",1779884494,3458611,"List(20.78, 930, 81, 1022, 1022, 20.55, 21.3, 19.69)",Liberdade,null,"List(BR, 8394, 1779874744, 1779913719, 1)",-10800,10000,"List(List(scattered clouds, 03d, 802, Clouds))","List(154, 4.02, 3.13)",2026,5,27,12


### Flatten nested JSON

In [0]:
df_flat = df_clean.select(
    col('coord.lat').alias('latitude'),
    col('coord.lon').alias('longitude'),
    col('main.temp').alias('temperature'),
    col('main.feels_like').alias('feels_like'),
    col('main.humidity').alias('humidity'),
    col('main.pressure').alias('pressure'),
    col('main.temp_max').alias('temp_max'),
    col('main.temp_min').alias('temp_min'),
    col('wind.speed').alias('wind_speed'),
    col('wind.deg').alias('wind_direction'),
    col('rain.`1h`').alias('rain_1h'),
    col('visibility'),
    from_unixtime(col('dt')).alias('event_time'),
    explode_outer(col('weather')).alias('weather_detail'),
    col('year'), col('month'), col('day'), col('hour')
)

In [0]:
df_flat.limit(2).display()

latitude,longitude,temperature,feels_like,humidity,pressure,temp_max,temp_min,wind_speed,wind_direction,rain_1h,visibility,event_time,weather_detail,year,month,day,hour
13.7563,100.5018,31.76,38.76,73,1006,32.95,31.07,5.46,192,null,10000,2026-05-27 12:18:23,"List(broken clouds, 04n, 803, Clouds)",2026,5,27,12
-23.5505,-46.6333,20.55,20.78,81,1022,21.3,19.69,3.13,154,null,10000,2026-05-27 12:21:34,"List(scattered clouds, 03d, 802, Clouds)",2026,5,27,12


### Extract from exploded weather

In [0]:
df_flat = df_flat.select(
    '*',
    col('weather_detail.main').alias('weather_main'),
    col('weather_detail.description').alias('weather_description'),
    col('weather_detail.id').alias('weather_code')
).drop('weather_detail')

# Convert event_time string to proper timestamp
df_flat = df_flat.withColumn('event_time', to_timestamp(col('event_time')))

In [0]:
df_flat.limit(12).display()

latitude,longitude,temperature,feels_like,humidity,pressure,temp_max,temp_min,wind_speed,wind_direction,rain_1h,visibility,event_time,year,month,day,hour,weather_main,weather_description,weather_code
13.7563,100.5018,31.76,38.76,73,1006,32.95,31.07,5.46,192,null,10000,2026-05-27T12:18:23.000Z,2026,5,27,12,Clouds,broken clouds,803
-23.5505,-46.6333,20.55,20.78,81,1022,21.3,19.69,3.13,154,null,10000,2026-05-27T12:21:34.000Z,2026,5,27,12,Clouds,scattered clouds,802
-33.8698,151.2083,18.87,18.9,80,1016,19.6,17.9,4.12,110,0.84,9000,2026-05-27T12:11:14.000Z,2026,5,27,12,Rain,shower rain,521
1.352,103.8198,29.74,36.35,78,1009,29.75,28.7,3.09,210,null,10000,2026-05-27T12:12:42.000Z,2026,5,27,12,Clouds,broken clouds,803
51.5072,-0.1276,26.13,26.13,50,1026,27.96,23.97,3.13,74,null,10000,2026-05-27T12:14:54.000Z,2026,5,27,12,Clear,clear sky,800
25.2048,55.2708,37.01,41.64,41,1005,38.18,37.01,5.14,270,null,10000,2026-05-27T12:21:00.000Z,2026,5,27,12,Clear,clear sky,800
40.7126,-74.01,21.41,21.59,76,1013,22.73,19.95,4.12,220,null,10000,2026-05-27T12:20:52.000Z,2026,5,27,12,Clouds,few clouds,801
48.8566,2.3522,31.14,30.54,36,1023,32.45,30.23,3.09,50,null,10000,2026-05-27T12:13:46.000Z,2026,5,27,12,Clear,clear sky,800
52.52,13.405,22.52,21.74,35,1023,23.32,21.19,5.14,330,null,10000,2026-05-27T12:16:29.000Z,2026,5,27,12,Clear,clear sky,800
19.076,72.8777,33.0,40.0,66,1008,33.0,32.95,5.66,260,null,7000,2026-05-27T12:13:01.000Z,2026,5,27,12,Haze,haze,721


In [0]:
df_flat.printSchema()

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- feels_like: double (nullable = true)
 |-- humidity: long (nullable = true)
 |-- pressure: long (nullable = true)
 |-- temp_max: double (nullable = true)
 |-- temp_min: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction: long (nullable = true)
 |-- rain_1h: double (nullable = true)
 |-- visibility: long (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- weather_main: string (nullable = true)
 |-- weather_description: string (nullable = true)
 |-- weather_code: long (nullable = true)



### Map lat/lon → Correct City Name

In [0]:
window_city = Window.partitionBy('latitude', 'longitude', 'event_time').orderBy('_dist')

df_mapped = df_flat \
    .crossJoin(broadcast(df_city_ref)) \
    .withColumn('_lat_diff', abs(col('latitude') - col('c_lat'))) \
    .withColumn('_lon_diff', abs(col('longitude') - col('c_lon'))) \
    .withColumn('_dist', col('_lat_diff') + col('_lon_diff')) \
    .filter(col('_dist') < 1.0) \
    .withColumn('_rn', row_number().over(window_city)) \
    .filter(col('_rn') == 1) \
    .drop('_lat_diff', '_lon_diff', '_dist', '_rn', 'c_lat', 'c_lon') \
    .withColumnRenamed('mapped_city', 'city') \
    .withColumnRenamed('mapped_country', 'country_code')

print(f'Rows after city mapping: {df_mapped.count()}')
df_mapped.select('city', 'country_code', 'event_time', 'latitude', 'longitude').distinct().show()

Rows after city mapping: 10
+---------+------------+-------------------+--------+---------+
|     city|country_code|         event_time|latitude|longitude|
+---------+------------+-------------------+--------+---------+
|   Sydney|          AU|2026-05-27 12:11:14|-33.8698| 151.2083|
|Sao Paulo|          BR|2026-05-27 12:21:34|-23.5505| -46.6333|
|Singapore|          SG|2026-05-27 12:12:42|   1.352| 103.8198|
|  Bangkok|          TH|2026-05-27 12:18:23| 13.7563| 100.5018|
|   Mumbai|          IN|2026-05-27 12:13:01|  19.076|  72.8777|
|    Dubai|          AE|2026-05-27 12:21:00| 25.2048|  55.2708|
| New York|          US|2026-05-27 12:20:52| 40.7126|   -74.01|
|    Paris|          FR|2026-05-27 12:13:46| 48.8566|   2.3522|
|   London|          GB|2026-05-27 12:14:54| 51.5072|  -0.1276|
|   Berlin|          DE|2026-05-27 12:16:29|   52.52|   13.405|
+---------+------------+-------------------+--------+---------+



### Deduplicate Weather Array Rows

In [0]:
window_dedup = Window.partitionBy('city', 'event_time').orderBy('weather_code')

df_silver = df_mapped \
    .withColumn('_rn', row_number().over(window_dedup)) \
    .filter(col('_rn') == 1) \
    .drop('_rn') \
    .withColumn('join_time', date_trunc('hour', col('event_time'))) \
    .fillna(0.0, subset=['rain_1h'])

print(f'Final silver rows: {df_silver.count()}')
df_silver.select('city', 'country_code', 'join_time', 'temperature', 'rain_1h', 'weather_main').show(10)

Final silver rows: 10
+---------+------------+-------------------+-----------+-------+------------+
|     city|country_code|          join_time|temperature|rain_1h|weather_main|
+---------+------------+-------------------+-----------+-------+------------+
|  Bangkok|          TH|2026-05-27 12:00:00|      31.76|    0.0|      Clouds|
|   Berlin|          DE|2026-05-27 12:00:00|      22.52|    0.0|       Clear|
|    Dubai|          AE|2026-05-27 12:00:00|      37.01|    0.0|       Clear|
|   London|          GB|2026-05-27 12:00:00|      26.13|    0.0|       Clear|
|   Mumbai|          IN|2026-05-27 12:00:00|       33.0|    0.0|        Haze|
| New York|          US|2026-05-27 12:00:00|      21.41|    0.0|      Clouds|
|    Paris|          FR|2026-05-27 12:00:00|      31.14|    0.0|       Clear|
|Sao Paulo|          BR|2026-05-27 12:00:00|      20.55|    0.0|      Clouds|
|Singapore|          SG|2026-05-27 12:00:00|      29.74|    0.0|      Clouds|
|   Sydney|          AU|2026-05-27 12:00:0

In [0]:
print('Distinct cities (should be 10):')
df_silver.select('city', 'country_code').distinct().orderBy('city').show()
print('Schema:')
df_silver.printSchema()

Distinct cities (should be 10):
+---------+------------+
|     city|country_code|
+---------+------------+
|  Bangkok|          TH|
|   Berlin|          DE|
|    Dubai|          AE|
|   London|          GB|
|   Mumbai|          IN|
| New York|          US|
|    Paris|          FR|
|Sao Paulo|          BR|
|Singapore|          SG|
|   Sydney|          AU|
+---------+------------+

Schema:
root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- feels_like: double (nullable = true)
 |-- humidity: long (nullable = true)
 |-- pressure: long (nullable = true)
 |-- temp_max: double (nullable = true)
 |-- temp_min: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction: long (nullable = true)
 |-- rain_1h: double (nullable = false)
 |-- visibility: long (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = tr

In [0]:
df_silver.limit(15).display()

latitude,longitude,temperature,feels_like,humidity,pressure,temp_max,temp_min,wind_speed,wind_direction,rain_1h,visibility,event_time,year,month,day,hour,weather_main,weather_description,weather_code,city,country_code,join_time
13.7563,100.5018,31.76,38.76,73,1006,32.95,31.07,5.46,192,0.0,10000,2026-05-27T12:18:23.000Z,2026,5,27,12,Clouds,broken clouds,803,Bangkok,TH,2026-05-27T12:00:00.000Z
52.52,13.405,22.52,21.74,35,1023,23.32,21.19,5.14,330,0.0,10000,2026-05-27T12:16:29.000Z,2026,5,27,12,Clear,clear sky,800,Berlin,DE,2026-05-27T12:00:00.000Z
25.2048,55.2708,37.01,41.64,41,1005,38.18,37.01,5.14,270,0.0,10000,2026-05-27T12:21:00.000Z,2026,5,27,12,Clear,clear sky,800,Dubai,AE,2026-05-27T12:00:00.000Z
51.5072,-0.1276,26.13,26.13,50,1026,27.96,23.97,3.13,74,0.0,10000,2026-05-27T12:14:54.000Z,2026,5,27,12,Clear,clear sky,800,London,GB,2026-05-27T12:00:00.000Z
19.076,72.8777,33.0,40.0,66,1008,33.0,32.95,5.66,260,0.0,7000,2026-05-27T12:13:01.000Z,2026,5,27,12,Haze,haze,721,Mumbai,IN,2026-05-27T12:00:00.000Z
40.7126,-74.01,21.41,21.59,76,1013,22.73,19.95,4.12,220,0.0,10000,2026-05-27T12:20:52.000Z,2026,5,27,12,Clouds,few clouds,801,New York,US,2026-05-27T12:00:00.000Z
48.8566,2.3522,31.14,30.54,36,1023,32.45,30.23,3.09,50,0.0,10000,2026-05-27T12:13:46.000Z,2026,5,27,12,Clear,clear sky,800,Paris,FR,2026-05-27T12:00:00.000Z
-23.5505,-46.6333,20.55,20.78,81,1022,21.3,19.69,3.13,154,0.0,10000,2026-05-27T12:21:34.000Z,2026,5,27,12,Clouds,scattered clouds,802,Sao Paulo,BR,2026-05-27T12:00:00.000Z
1.352,103.8198,29.74,36.35,78,1009,29.75,28.7,3.09,210,0.0,10000,2026-05-27T12:12:42.000Z,2026,5,27,12,Clouds,broken clouds,803,Singapore,SG,2026-05-27T12:00:00.000Z
-33.8698,151.2083,18.87,18.9,80,1016,19.6,17.9,4.12,110,0.84,9000,2026-05-27T12:11:14.000Z,2026,5,27,12,Rain,shower rain,521,Sydney,AU,2026-05-27T12:00:00.000Z


### Write to Silver Delta Lake

In [0]:
df_silver.write.format('delta').mode('overwrite').partitionBy("year", "month", "day").save(SILVER)